# NHIS Part I: Income and Interview Process Features

The baseline adult file now has a clean target and a first set of candidate predictors. The next step adds two support layers: imputed income and selected paradata variables.

The imputed income file strengthens the socioeconomic side of the project. The paradata file adds interview context, such as mode, timing, language, and contact attempts. These variables will be reviewed carefully before modeling. The goal is to keep variables that add useful context without making the project harder to explain.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## File paths

The project uses the same folder structure in PyCharm and Colab. The Colab code is included as a comment for later use.

In [2]:
# Colab option, leave commented out when working locally in PyCharm.
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = Path("/content/drive/MyDrive/med_project")

# Local/PyCharm option.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

DATA_INTERIM.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", DATA_RAW)

Project root: C:\Users\Alexi\projects\med_project
Raw data folder: C:\Users\Alexi\projects\med_project\data\raw


## Confirm available source files

Before merging, the raw data folder is checked so the file names can be verified. This helps avoid silent errors from loading the wrong file.

In [3]:
for path in sorted(DATA_RAW.iterdir()):
    print(path.name)

meps_2023_hc251_full_year_consolidated.xlsx
nhis_2023_paradata.csv
nhis_2023_sample_adult.csv
nhis_2023_sample_adult_imputed_income.csv


## Load the cleaned adult file and raw support files

The cleaned candidate file keeps the stable target and first-pass predictors. The raw adult file is loaded only to confirm identifiers and compare merge counts if needed.

Update the income file name below if your local file uses a different name.

In [4]:
adult_raw_path = DATA_RAW / "nhis_2023_sample_adult.csv"
adult_clean_path = DATA_INTERIM / "nhis_first_pass_cleaned_candidates.csv"
income_path = DATA_RAW / "nhis_2023_sample_adult_imputed_income.csv"
paradata_path = DATA_RAW / "nhis_2023_paradata.csv"

adult_raw = pd.read_csv(adult_raw_path, low_memory=False)
adult_clean = pd.read_csv(adult_clean_path, low_memory=False)
income = pd.read_csv(income_path, low_memory=False)
paradata = pd.read_csv(paradata_path, low_memory=False)

print("Adult raw shape:", adult_raw.shape)
print("Adult cleaned candidate shape:", adult_clean.shape)
print("Income shape:", income.shape)
print("Paradata shape:", paradata.shape)

Adult raw shape: (29522, 647)
Adult cleaned candidate shape: (28777, 62)
Income shape: (295220, 7)
Paradata shape: (63591, 123)


## Check merge identifiers

The adult, income, and paradata files should share the household identifier `HHX`. Before merging, each file is checked for missing identifiers and duplicate identifiers.

In [5]:
merge_files = {
    "adult_clean": adult_clean,
    "income": income,
    "paradata": paradata,
}

for name, df in merge_files.items():
    print("\n" + "=" * 80)
    print(name)

    if "HHX" not in df.columns:
        print("HHX is missing.")
        continue

    print("Rows:", len(df))
    print("Missing HHX:", df["HHX"].isna().sum())
    print("Unique HHX:", df["HHX"].nunique(dropna=False))
    print("Duplicate HHX rows:", df.duplicated(subset=["HHX"]).sum())


adult_clean
Rows: 28777
Missing HHX: 0
Unique HHX: 28777
Duplicate HHX rows: 0

income
Rows: 295220
Missing HHX: 0
Unique HHX: 29522
Duplicate HHX rows: 265698

paradata
Rows: 63591
Missing HHX: 0
Unique HHX: 63591
Duplicate HHX rows: 0


## Inspect income columns

The imputed income file may include several versions of income and poverty variables. The first pass keeps poverty category and income-to-poverty information because those are easiest to explain for a cost-related access question.

In [6]:
income.columns.tolist()

['RATCAT_A',
 'INCTCFLG_A',
 'IMPNUM_A',
 'IMPINCFLG_A',
 'RECTYPE',
 'POVRATTC_A',
 'HHX']

In [7]:
income_keyword_cols = [
    col for col in income.columns
    if any(keyword in col.upper() for keyword in ["HHX", "INC", "POV", "RAT", "IMP"])
]

income_keyword_cols

['RATCAT_A', 'INCTCFLG_A', 'IMPNUM_A', 'IMPINCFLG_A', 'POVRATTC_A', 'HHX']

## Select income variables

Poverty category should be the main feature. Continuous income-to-poverty variables can be kept for sensitivity checks but should not replace the categorical poverty measure in the first model.

In [8]:
preferred_income_cols = [
    "HHX",
    "RATCAT_A",
    "POVRATTC_A",
    "INCTCFLG_A",
    "IMPINCFLG_A",
]

income_keep = [col for col in preferred_income_cols if col in income.columns]

print("Income columns kept:", income_keep)
print("Missing preferred income columns:", [col for col in preferred_income_cols if col not in income.columns])

income_selected = income[income_keep].copy()

income_selected.head()

Income columns kept: ['HHX', 'RATCAT_A', 'POVRATTC_A', 'INCTCFLG_A', 'IMPINCFLG_A']
Missing preferred income columns: []


,HHX,RATCAT_A,POVRATTC_A,INCTCFLG_A,IMPINCFLG_A
0,H029691,4,1.01,0,0
1,H029691,4,1.01,0,0
2,H029691,4,1.01,0,0
3,H029691,4,1.01,0,0
4,H029691,4,1.01,0,0


## Prepare imputed income for merging

The imputed income file has multiple rows per adult because it contains repeated income imputations. A direct merge would duplicate the adult records, so the file needs to be summarized to one row per `HHX` before merging.

For the first model-ready version, poverty category is treated as the main socioeconomic feature. The continuous poverty ratio is averaged across imputations and kept as a sensitivity feature. A fuller publication version would analyze each imputed dataset separately and pool results, but the one-row summary is a practical and transparent first step for the course project.

In [9]:
print("Income imputation counts:")
print(income["IMPNUM_A"].value_counts(dropna=False).sort_index())

print("\nRows per HHX in income file:")
print(income.groupby("HHX").size().value_counts().sort_index())

Income imputation counts:
IMPNUM_A
1     29522
2     29522
3     29522
4     29522
5     29522
6     29522
7     29522
8     29522
9     29522
10    29522
Name: count, dtype: int64

Rows per HHX in income file:
10    29522
Name: count, dtype: int64


## Summarize income to one row per adult

The modal value is used for categorical income fields. The mean is used for the continuous poverty ratio. This keeps the merge one-to-one while preserving the main socioeconomic information.

In [10]:
def mode_or_nan(series):
    modes = series.dropna().mode()
    if modes.empty:
        return np.nan
    return modes.iloc[0]


income_summary = (
    income
    .groupby("HHX", as_index=False)
    .agg(
        RATCAT_A_imp=("RATCAT_A", mode_or_nan),
        POVRATTC_A_imp_mean=("POVRATTC_A", "mean"),
        POVRATTC_A_imp_min=("POVRATTC_A", "min"),
        POVRATTC_A_imp_max=("POVRATTC_A", "max"),
        INCTCFLG_A_imp=("INCTCFLG_A", mode_or_nan),
        IMPINCFLG_A_imp=("IMPINCFLG_A", mode_or_nan),
        income_imputation_count=("IMPNUM_A", "count"),
    )
)

print("Income summary shape:", income_summary.shape)
income_summary.head()

Income summary shape: (29522, 8)


,HHX,RATCAT_A_imp,POVRATTC_A_imp_mean,POVRATTC_A_imp_min,POVRATTC_A_imp_max,INCTCFLG_A_imp,IMPINCFLG_A_imp,income_imputation_count
0,H000002,12,4.09,4.09,4.09,0,0,10
1,H000005,6,1.27,0.10,2.13,0,2,10
2,H000006,11,3.94,3.94,3.94,0,0,10
3,H000007,11,3.74,3.74,3.74,0,0,10
4,H000008,4,1.13,1.13,1.13,0,0,10


## Check income summary

The summarized income file should now have one row per adult. The poverty category should be the main feature for modeling because it is easier to explain than a continuous poverty ratio.

In [11]:
print("Unique HHX:", income_summary["HHX"].nunique())
print("Duplicate HHX rows:", income_summary.duplicated(subset=["HHX"]).sum())

print("\nPoverty category distribution:")
print(income_summary["RATCAT_A_imp"].value_counts(dropna=False).sort_index())

print("\nContinuous poverty ratio summary:")
print(income_summary["POVRATTC_A_imp_mean"].describe())

Unique HHX: 29522
Duplicate HHX rows: 0

Poverty category distribution:
RATCAT_A_imp
1      877
2      944
3     1364
4     1160
5     1582
6     1220
7     1375
8     2527
9     2561
10    1862
11    1647
12    1464
13    1216
14    9723
Name: count, dtype: int64

Continuous poverty ratio summary:
count    29522.000000
mean         4.113444
std          2.866666
min          0.000000
25%          1.868000
50%          3.390000
75%          5.650000
max         11.000000
Name: POVRATTC_A_imp_mean, dtype: float64


## Merge income with the cleaned adult file

The cleaned adult file has one row per adult. The summarized income file also has one row per adult, so the merge should validate as one-to-one.

In [12]:
adult_income = adult_clean.merge(
    income_summary,
    on="HHX",
    how="left",
    validate="one_to_one"
)

print("Adult cleaned shape:", adult_clean.shape)
print("Adult + income shape:", adult_income.shape)

income_added_cols = [
    "RATCAT_A_imp",
    "POVRATTC_A_imp_mean",
    "POVRATTC_A_imp_min",
    "POVRATTC_A_imp_max",
    "INCTCFLG_A_imp",
    "IMPINCFLG_A_imp",
    "income_imputation_count",
]

print("\nMissing values in added income columns:")
print(adult_income[income_added_cols].isna().sum())

Adult cleaned shape: (28777, 62)
Adult + income shape: (28777, 69)

Missing values in added income columns:
RATCAT_A_imp               0
POVRATTC_A_imp_mean        0
POVRATTC_A_imp_min         0
POVRATTC_A_imp_max         0
INCTCFLG_A_imp             0
IMPINCFLG_A_imp            0
income_imputation_count    0
dtype: int64


## Compare adult-file poverty variables with imputed income variables

The adult file already contains poverty variables. The imputed income file provides a more explicit income-imputation layer. Comparing the two helps decide whether to keep both, replace the adult-file version, or use the imputed version as the main socioeconomic feature.

In [13]:
compare_income_cols = [
    "RATCAT_A",
    "RATCAT_A_imp",
    "POVRATTC_A",
    "POVRATTC_A_imp_mean",
    "INCTCFLG_A",
    "INCTCFLG_A_imp",
    "IMPINCFLG_A",
    "IMPINCFLG_A_imp",
]

compare_income_cols = [col for col in compare_income_cols if col in adult_income.columns]

adult_income[compare_income_cols].head()

,RATCAT_A,RATCAT_A_imp,POVRATTC_A,POVRATTC_A_imp_mean,INCTCFLG_A,INCTCFLG_A_imp,IMPINCFLG_A,IMPINCFLG_A_imp
0,4,4,1.01,1.01,0,0,0,0
1,8,8,2.49,2.49,0,0,0,0
2,14,14,6.73,6.73,0,0,0,0
3,10,10,3.43,3.43,0,0,0,0
4,5,5,1.27,1.27,0,0,0,0


In [14]:
if {"RATCAT_A", "RATCAT_A_imp"}.issubset(adult_income.columns):
    print("RATCAT agreement rate:")
    print((adult_income["RATCAT_A"] == adult_income["RATCAT_A_imp"]).mean())

if {"POVRATTC_A", "POVRATTC_A_imp_mean"}.issubset(adult_income.columns):
    diff = adult_income["POVRATTC_A"] - adult_income["POVRATTC_A_imp_mean"]
    print("\nPOVRATTC difference summary:")
    print(diff.describe())

RATCAT agreement rate:
0.9172255620808284

POVRATTC difference summary:
count    28777.000000
mean        -0.007459
std          0.709190
min         -6.634000
25%          0.000000
50%          0.000000
75%          0.000000
max          8.148000
dtype: float64


## Select paradata variables

The paradata file contains interview process information. These variables can add context about how the interview was completed, but they should not overpower the health and access variables from the Sample Adult file.

The first pass keeps a small set of paradata variables related to mode, language, interpreter use, timing, and contact effort. Final interview outcome variables are excluded because they describe survey completion rather than health care access.

In [15]:
paradata.columns.tolist()

['SRVY_YR',
 'CTSTAT3',
 'CTSTAT2',
 'CTSTAT1',
 'AD_PCLASS',
 'HH_PCLASS',
 'QUALITY_SC',
 'QUALITY_SA',
 'UNABLE5R',
 'UNABLE4R',
 'UNABLE3R',
 'UNABLE2R',
 'UNABLE1R',
 'STRT14R',
 'STRAT13R',
 'STRAT12R',
 'STRT11R',
 'STRAT06R',
 'STRAT05R',
 'STRAT04R',
 'STRAT03R',
 'STRAT02R',
 'STRAT01R',
 'RELC15R',
 'RELUC12R',
 'RELUC11R',
 'RELC09R',
 'RELC08R',
 'RELUC07R',
 'RELUC06R',
 'RELUC05R',
 'RELUC03R',
 'RELUC02R',
 'RELC01R',
 'NCTP12R',
 'NCTPR11R',
 'NCTPR10R',
 'NCTPR09R',
 'NCTPR08R',
 'NCTPR07R',
 'NCTP05R',
 'NCTPR04R',
 'NCTPR03R',
 'NCTP01R',
 'NCTL07R',
 'NCTL06R',
 'NCTEL05R',
 'NCTEL04R',
 'NCTEL03R',
 'NCTL02R',
 'NCTL01R',
 'MODE_T',
 'MODE_P',
 'LANG5R',
 'LANG4R',
 'LANG3R',
 'LANG2R',
 'LANG1R',
 'TOTCOUNT',
 'STRAT99R',
 'STRAT98R',
 'RELUC99R',
 'RELUC98R',
 'NCTEL99R',
 'NCTPR99R',
 'UNABL99R',
 'REASSIGN',
 'RLINK_C',
 'RLINK_A',
 'SC_TOD',
 'SCSTRPNT',
 'PHONELIVE_C',
 'TELCURWRK_C',
 'SA_TOD',
 'SASTRPNT',
 'PHONEUSE_A',
 'PHONELIVE_A',
 'TELCEL_A',
 'TELC

In [16]:
paradata_keyword_cols = [
    col for col in paradata.columns
    if any(keyword in col.upper() for keyword in [
        "HHX",
        "MODE",
        "LANG",
        "RPT",
        "TIME",
        "TOD",
        "STR",
        "COUNT",
        "QUALITY",
        "CONTACT"
    ])
]

paradata_keyword_cols

['QUALITY_SC',
 'QUALITY_SA',
 'STRT14R',
 'STRAT13R',
 'STRAT12R',
 'STRT11R',
 'STRAT06R',
 'STRAT05R',
 'STRAT04R',
 'STRAT03R',
 'STRAT02R',
 'STRAT01R',
 'MODE_T',
 'MODE_P',
 'LANG5R',
 'LANG4R',
 'LANG3R',
 'LANG2R',
 'LANG1R',
 'TOTCOUNT',
 'STRAT99R',
 'STRAT98R',
 'SC_TOD',
 'SCSTRPNT',
 'SA_TOD',
 'SASTRPNT',
 'STRTPNT',
 'HHC_TOD',
 'INTMODEWHYSC',
 'INTMODESC',
 'INTMODEWHYSA',
 'INTMODESA',
 'INTRPTSC',
 'INTLANGSC',
 'INTRPTSA',
 'INTLANGSA',
 'PSTRAT',
 'HHX']

## Keep selected paradata fields

The selected variables are intentionally limited. They should help describe interview context without creating leakage from final survey disposition fields.

In [17]:
preferred_paradata_cols = [
    "HHX",
    "MODE_P",
    "MODE_T",
    "TOTCOUNT",
    "INTMODESA",
    "INTMODEWHYSA",
    "INTLANGSA",
    "INTRPTSA",
    "SASTRPNT",
    "SA_TOD",
    "QUALITY_SA",
]

paradata_keep = [col for col in preferred_paradata_cols if col in paradata.columns]

print("Paradata columns kept:", paradata_keep)
print("Missing preferred paradata columns:", [col for col in preferred_paradata_cols if col not in paradata.columns])

paradata_selected = paradata[paradata_keep].copy()

paradata_selected.head()

Paradata columns kept: ['HHX', 'MODE_P', 'MODE_T', 'TOTCOUNT', 'INTMODESA', 'INTMODEWHYSA', 'INTLANGSA', 'INTRPTSA', 'SASTRPNT', 'SA_TOD', 'QUALITY_SA']
Missing preferred paradata columns: []


,HHX,MODE_P,MODE_T,TOTCOUNT,INTMODESA,INTMODEWHYSA,INTLANGSA,INTRPTSA,SASTRPNT,SA_TOD,QUALITY_SA
0,H003873,0.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,H029691,1.0,0.0,1.0,3.0,NaN,1.0,NaN,1.0,2.0,NaN
2,H028812,1.0,0.0,1.0,3.0,NaN,1.0,NaN,1.0,2.0,NaN
3,H032265,5.0,1.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,H045277,4.0,1.0,5.0,3.0,NaN,1.0,NaN,3.0,1.0,NaN


## Check selected paradata fields

The selected paradata file should have one row per `HHX`. Missing values after merging will show whether every adult in the cleaned analytic sample has matching paradata.

In [18]:
print("Selected paradata shape:", paradata_selected.shape)
print("Unique HHX:", paradata_selected["HHX"].nunique())
print("Duplicate HHX rows:", paradata_selected.duplicated(subset=["HHX"]).sum())

for col in paradata_keep:
    if col == "HHX":
        continue

    print("\n" + "=" * 80)
    print(col)

    if paradata_selected[col].nunique(dropna=False) <= 30:
        print(paradata_selected[col].value_counts(dropna=False).sort_index())
    else:
        print(paradata_selected[col].describe())

Selected paradata shape: (63591, 11)
Unique HHX: 63591
Duplicate HHX rows: 0

MODE_P
MODE_P
0.0      1811
1.0     16987
2.0     16035
3.0     11501
4.0      7011
5.0      4227
6.0      2392
7.0      1371
8.0       777
9.0       468
10.0      272
11.0      193
12.0       92
13.0       69
14.0       41
15.0       18
16.0       14
17.0        7
18.0        8
19.0        3
20.0        1
21.0        2
22.0        3
24.0        4
NaN       284
Name: count, dtype: int64

MODE_T
count    63307.000000
mean         2.186709
std          2.598476
min          0.000000
25%          0.000000
50%          1.000000
75%          3.000000
max         46.000000
Name: MODE_T, dtype: float64

TOTCOUNT
count    63315.000000
mean         5.084309
std          3.558215
min          1.000000
25%          2.000000
50%          4.000000
75%          7.000000
max         49.000000
Name: TOTCOUNT, dtype: float64

INTMODESA
INTMODESA
1.0    15369
2.0     1001
3.0    13826
NaN    33395
Name: count, dtype: int64

IN

## Merge paradata with adult and income file

The adult-income file has one row per adult. The selected paradata file also has one row per household identifier, so the merge should validate as one-to-one.

In [19]:
adult_income_paradata = adult_income.merge(
    paradata_selected,
    on="HHX",
    how="left",
    validate="one_to_one"
)

print("Adult + income shape:", adult_income.shape)
print("Adult + income + paradata shape:", adult_income_paradata.shape)

paradata_added_cols = [col for col in paradata_keep if col != "HHX"]

print("\nMissing values in added paradata columns:")
print(adult_income_paradata[paradata_added_cols].isna().sum())

Adult + income shape: (28777, 69)
Adult + income + paradata shape: (28777, 79)

Missing values in added paradata columns:
MODE_P             29
MODE_T             29
TOTCOUNT           29
INTMODESA           0
INTMODEWHYSA    13151
INTLANGSA           0
INTRPTSA        27306
SASTRPNT           31
SA_TOD             32
QUALITY_SA      28777
dtype: int64


## Create simple paradata features

Contact effort and mode are easier to interpret as proportions than as raw counts alone. The first pass creates proportions for personal visit attempts and telephone attempts when total contact count is available.

In [20]:
if {"MODE_P", "TOTCOUNT"}.issubset(adult_income_paradata.columns):
    adult_income_paradata["personal_visit_contact_share"] = np.where(
        adult_income_paradata["TOTCOUNT"] > 0,
        adult_income_paradata["MODE_P"] / adult_income_paradata["TOTCOUNT"],
        np.nan
    )

if {"MODE_T", "TOTCOUNT"}.issubset(adult_income_paradata.columns):
    adult_income_paradata["telephone_contact_share"] = np.where(
        adult_income_paradata["TOTCOUNT"] > 0,
        adult_income_paradata["MODE_T"] / adult_income_paradata["TOTCOUNT"],
        np.nan
    )

created_paradata_features = [
    col for col in [
        "personal_visit_contact_share",
        "telephone_contact_share"
    ]
    if col in adult_income_paradata.columns
]

adult_income_paradata[created_paradata_features].describe()

,personal_visit_contact_share,telephone_contact_share
count,28748.000000,28748.000000
mean,0.640961,0.343559
std,0.309878,0.300497
min,0.000000,0.000000
25%,0.400000,0.000000
50%,0.625000,0.333333
75%,1.000000,0.571429
max,1.000000,1.000000


## Review missingness after all merges

The merged file is still an interim analytic file. The next modeling version can decide whether the income and paradata features improve the baseline file enough to keep.

In [21]:
merged_missing = (
    adult_income_paradata
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "variable", 0: "missing_count"})
)

merged_missing["missing_rate"] = merged_missing["missing_count"] / len(adult_income_paradata)

merged_missing = merged_missing.sort_values("missing_rate", ascending=False)

merged_missing.head(30)

,variable,missing_count,missing_rate
78,QUALITY_SA,28777,1.000000
75,INTRPTSA,27306,0.948883
53,PAYNOBLLNW_A,25938,0.901345
60,WELLVIS_A,23693,0.823331
48,COVER65_A,19338,0.671995
73,INTMODEWHYSA,13151,0.456997
47,COVER_A,9470,0.329082
58,USPLKIND_A,2680,0.093130
11,EDUCP_A,130,0.004517
51,PRIVATE_A,110,0.003822


## Final merge cleanup decisions

The merged file includes the cleaned adult variables, summarized income variables, selected paradata fields, and derived contact-share features. A few paradata variables are removed before saving because they do not add usable information for the first modeling version.

`QUALITY_SA` is fully missing in the analytic sample. `INTRPTSA` is also too sparse for the first model-ready version. These variables are removed before saving the cleaned merged file.

`INTMODEWHYSA` is kept for now because it may describe why the Sample Adult interview was conducted by phone, but it should be reviewed again before modeling.

In [24]:
drop_from_merged = [
    "QUALITY_SA",
    "INTRPTSA",
]

drop_from_merged = [
    col for col in drop_from_merged
    if col in adult_income_paradata.columns
]

adult_income_paradata = adult_income_paradata.drop(columns=drop_from_merged)

print("Dropped from merged file:", drop_from_merged)
print("Clean merged shape:", adult_income_paradata.shape)

Dropped from merged file: ['QUALITY_SA', 'INTRPTSA']
Clean merged shape: (28777, 79)


## Missingness after final merge cleanup

The cleaned merged file should keep useful income and paradata variables while removing fields that are fully missing or too sparse for the first modeling version. Remaining missingness will be handled in the modeling pipeline or reviewed during feature selection.

In [25]:
merged_clean_missing = (
    adult_income_paradata
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "variable", 0: "missing_count"})
)

merged_clean_missing["missing_rate"] = (
    merged_clean_missing["missing_count"] / len(adult_income_paradata)
)

merged_clean_missing = merged_clean_missing.sort_values("missing_rate", ascending=False)

merged_clean_missing.head(25)

,variable,missing_count,missing_rate
53,PAYNOBLLNW_A,25938,0.901345
60,WELLVIS_A,23693,0.823331
48,COVER65_A,19338,0.671995
73,INTMODEWHYSA,13151,0.456997
47,COVER_A,9470,0.329082
58,USPLKIND_A,2680,0.093130
11,EDUCP_A,130,0.004517
51,PRIVATE_A,110,0.003822
49,MEDICARE_A,88,0.003058
35,CHDEV_A,82,0.002849


## Save cleaned merged NHIS file

This file is the cleaned merged analytic version for Part I. The modeling notebook will decide which baseline, income, and paradata features are included in the final predictor set.

In [26]:
output_path = DATA_INTERIM / "nhis_adult_income_paradata_merged_clean.csv"

adult_income_paradata.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Saved shape:", adult_income_paradata.shape)

Saved: C:\Users\Alexi\projects\med_project\data\interim\nhis_adult_income_paradata_merged_clean.csv
Saved shape: (28777, 79)
